# 03 – MRI analysis

Analysis of the MRI scans (3D T1-weighted and 3D T2-weighted) of phantom A3.

The notebook follows the same structure as the CT notebook:

| Section | Content | Guide |
|---|---|---|
| 0 | Setup, data and acquisition parameters | – |
| 1 | Locating and cropping phantom A3 | – |
| 2 | Number, geometry and arrangement of objects | 1.1 |
| 3 | Internal structure and material characteristics | 1.2 |
| 4 | Contrast and image quality | 1.3 |
| 5 | Summary for the multimodal comparison | 1.4–1.5 |
| 6 | Reproducibility record | – |

Each analysis follows **question → method → result → interpretation/limitations**. Coordinates for ROIs and measurement lines are chosen by the group in the viewer and written in the cells marked *Placeholder*.

---
---
## 0. Setup and data

In [7]:
# === Autoreload: pick up changes in src/ automatically ===
%load_ext autoreload
%autoreload 2

# === Imports ===
import sys
sys.path.append("..")

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage

# === Our own functions ===
from src.data_io import load_series, save_table, save_figure
from src.viewers import viewer, plane_names, first_header, show_slice, contour_mm
from src.html_viewer import html_viewer
from src.measure import roi_stats, roi_stability, line_profile, width_crossing, edge_width

# === Paths ===
RAW       = Path("../data/raw")
PROCESSED = Path("../data/processed")
RESULTS   = Path("../results")

MRI_T1 = RAW / "MRI/3D_T1w"
MRI_T2 = RAW / "MRI/3D_T2w"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### What the MRI signal depends on

Unlike CT, MRI intensities are **not calibrated**. They are given in arbitrary units (a.u.) and depend on the sequence, coil and scanner settings, so values can only be compared within the same image, not between T1w and T2w or with CT.

For a spin-echo sequence, the signal from a voxel is approximately

$$S \propto \rho \cdot \left(1 - e^{-TR/T_1}\right) \cdot e^{-TE/T_2}$$

- $\rho$: proton density, i.e. how many MR-visible protons there are
- $T_1$, $T_2$: relaxation times of the material (material properties)
- $TR$, $TE$: repetition and echo time (chosen by the operator)

A **T1-weighted** image uses short TR and short TE, so differences in $T_1$ dominate. A **T2-weighted** image uses long TR and long TE, so differences in $T_2$ dominate.

Materials with very few mobile protons or a very short $T_2$ (air, dense solids) give almost no signal in both, so they appear dark.

### Load MRI data

Both series are loaded as 3D volumes, sorted by slice position. The slice spacing is calculated from the slice positions in the DICOM headers, not from `SliceThickness`.

In [8]:
t1, t1_spacing = load_series(MRI_T1)
t2, t2_spacing = load_series(MRI_T2)

h1 = first_header(MRI_T1)
h2 = first_header(MRI_T2)

t1_names = plane_names(h1)
t2_names = plane_names(h2)

for name, vol, sp, names, h in [("T1w", t1, t1_spacing, t1_names, h1),
                                 ("T2w", t2, t2_spacing, t2_names, h2)]:
    print(f"{name}:")
    print(f"  Shape (slices, rows, cols): {vol.shape}")
    print(f"  Voxel spacing (mm): {tuple(round(s, 3) for s in sp)}")
    print(f"  Slice thickness in header: {h.get('SliceThickness')} mm  |  slice spacing from positions: {sp[0]:.3f} mm")
    print(f"  Planes: {names}")

T1w:
  Shape (slices, rows, cols): (120, 512, 512)
  Voxel spacing (mm): (0.6, 0.273, 0.273)
  Slice thickness in header: 1.2 mm  |  slice spacing from positions: 0.600 mm
  Planes: ('Sagittal', 'Axial', 'Coronal')
T2w:
  Shape (slices, rows, cols): (120, 256, 256)
  Voxel spacing (mm): (0.6, 0.547, 0.547)
  Slice thickness in header: 1.2 mm  |  slice spacing from positions: 0.600 mm
  Planes: ('Sagittal', 'Axial', 'Coronal')


### Acquisition parameters

The sequence parameters are collected for the Materials and methods section. TR and TE show whether each image is really T1- or T2-weighted.

In [9]:
def mri_params(h):
    # Key acquisition parameters from a DICOM header
    return {
        "Manufacturer":            h.get("Manufacturer"),
        "Model":                   h.get("ManufacturerModelName"),
        "Field strength (T)":      h.get("MagneticFieldStrength"),
        "Sequence name":           h.get("SequenceName"),
        "Scanning sequence":       h.get("ScanningSequence"),
        "TR (ms)":                 h.get("RepetitionTime"),
        "TE (ms)":                 h.get("EchoTime"),
        "Flip angle (°)":          h.get("FlipAngle"),
        "Slice thickness (mm)":    h.get("SliceThickness"),
        "Pixel spacing (mm)":      h.get("PixelSpacing"),
        "Matrix (rows x cols)":    f"{h.get('Rows')} x {h.get('Columns')}",
        "Patient position":        h.get("PatientPosition"),
        "Image orientation":       h.get("ImageOrientationPatient"),
    }

acq = pd.DataFrame({"T1w": mri_params(h1), "T2w": mri_params(h2)})
display(acq)

,T1w,T2w
Manufacturer,GE MEDICAL SYSTEMS,GE MEDICAL SYSTEMS
Model,SIGNA Premier,SIGNA Premier
Field strength (T),3.0,3.0
Sequence name,None,None
Scanning sequence,SE,SE
TR (ms),999.0,4303.0
TE (ms),9.526,94.991
Flip angle (°),90.0,90.0
Slice thickness (mm),1.2,1.2
Pixel spacing (mm),"[0.2734, 0.2734]","[0.5469, 0.5469]"


> **Notes on the parameters:**
> - Are TR and TE short/short for T1w and long/long for T2w?
> - Missing values (`None`): ...

### Overview of the full volumes

The MRI scans contain two phantoms (3A and 3B). Scroll through both series to see how the phantoms are placed and how the objects appear.

In [10]:
html_viewer(t1, t1_spacing, names=t1_names, title="MRI T1w - full volume", modality="MR")
html_viewer(t2, t2_spacing, names=t2_names, title="MRI T2w - full volume", modality="MR")